In [1]:
import pandas as pd

# Load both CSVs (Recency-based and TF–IDF-based)
recency_path = "user_product_scores_for_review.csv"
tfidf_path = "user_product_tfidf_scores.csv"

recency = pd.read_csv(recency_path)
tfidf = pd.read_csv(tfidf_path)

print(f"Loaded {len(recency):,} rows (Recency) and {len(tfidf):,} rows (TF–IDF)")

# Merge on user_id + product_id for direct comparison
comparison = pd.merge(
    recency.rename(columns={'score': 'recency_score'}),
    tfidf[['user_id', 'product_id', 'tfidf_score']],
    on=['user_id', 'product_id'],
    how='outer'
).fillna(0)

print(f"Merged shape: {comparison.shape}")
display(comparison.head())

Loaded 13,307,953 rows (Recency) and 13,307,953 rows (TF–IDF)
Merged shape: (13307953, 4)


,user_id,product_id,recency_score,tfidf_score
0,1,196,4.192653,0.550754
1,1,10258,4.304592,0.902148
2,1,10326,0.398121,0.079237
3,1,12427,4.570855,0.815371
4,1,13032,1.765810,0.258171


### Comparing Recency vs TF–IDF Scores
Each user–product pair now contains two types of scores:
- **Recency-based score** → measures how frequently and recently the product was bought.  
- **TF–IDF score** → measures how distinctive the product is for the user compared to all users.  

In [2]:
# Calculate difference and ratio between methods
comparison['diff'] = comparison['tfidf_score'] - comparison['recency_score']
comparison['ratio'] = comparison['tfidf_score'] / (comparison['recency_score'] + 1e-9)

print("\nSample comparison:")
display(comparison[['user_id', 'product_id', 'recency_score', 'tfidf_score', 'diff', 'ratio']].head(10))


Sample comparison:


,user_id,product_id,recency_score,tfidf_score,diff,ratio
0,1,196,4.192653,0.550754,-3.641899,0.131362
1,1,10258,4.304592,0.902148,-3.402444,0.209578
2,1,10326,0.398121,0.079237,-0.318883,0.199028
3,1,12427,4.570855,0.815371,-3.755484,0.178385
4,1,13032,1.765810,0.258171,-1.507639,0.146205
5,1,13176,0.663617,0.039907,-0.623710,0.060136
6,1,14084,0.193786,0.071632,-0.122154,0.369644
7,1,17122,0.415198,0.064816,-0.350382,0.156109
8,1,25133,4.160838,0.657987,-3.502850,0.158138
9,1,26088,0.410647,0.175522,-0.235125,0.427427


### Interpretation of Metrics
- **diff** → Positive values mean TF–IDF assigns more importance (rare or unique products).  
- **ratio** → How much TF–IDF amplifies or reduces the recency weight.  

In [3]:
# Descriptive statistics across all pairs
print("\nDescriptive stats:")
display(comparison[['recency_score', 'tfidf_score', 'diff', 'ratio']].describe())


Descriptive stats:


,recency_score,tfidf_score,diff,ratio
count,1.330795e+07,1.330795e+07,1.330795e+07,1.330795e+07
mean,6.788598e-01,7.044171e-02,-6.084180e-01,5.007972e+02
std,1.007330e+00,1.200172e-01,9.588849e-01,4.858014e+04
min,9.657902e-15,4.025991e-04,-8.277644e+01,4.776801e-04
25%,1.057571e-01,1.664217e-02,-7.930013e-01,5.305149e-02
50%,4.252832e-01,3.629646e-02,-3.617264e-01,1.290811e-01
75%,8.782484e-01,7.941942e-02,-7.302294e-02,3.341903e-01
max,8.345534e+01,1.044489e+01,6.517350e+00,2.387426e+07


**In short:**  
- **Recency** focuses on *how often and how recently* the user bought something.  
- **TF–IDF** focuses on *how special* a product is to that user compared to everyone else.  
Comparing both gives insight into whether your recommendation model should emphasize **personal uniqueness** or **habitual frequency**.

In [4]:
# Identify biggest changes between the two methods
print("\nTop 10 products with largest positive TF–IDF lift:")
display(comparison.sort_values('diff', ascending=False).head(10)[['user_id', 'product_id', 'recency_score', 'tfidf_score', 'diff']])

print("\nTop 10 products most down-weighted by TF–IDF:")
display(comparison.sort_values('diff', ascending=True).head(10)[['user_id', 'product_id', 'recency_score', 'tfidf_score', 'diff']])


Top 10 products with largest positive TF–IDF lift:


,user_id,product_id,recency_score,tfidf_score,diff
4278734,66343,47210,3.773385,10.290735,6.517350
11835303,183315,15015,2.912479,8.770910,5.858431
10774343,166751,49001,2.970446,8.802658,5.832213
5440212,84062,37070,2.869929,8.324622,5.454694
1458516,22711,44573,2.979110,7.793994,4.814884
5994945,92559,1078,3.917324,8.429983,4.512659
7844861,121129,22683,2.949822,7.432624,4.482802
2503075,38979,13128,2.968774,7.440855,4.472081
5518325,85230,45328,2.964251,7.313804,4.349553
6258282,96613,48238,2.989555,7.323991,4.334436



Top 10 products most down-weighted by TF–IDF:


,user_id,product_id,recency_score,tfidf_score,diff
4517673,69919,24852,83.455343,0.678899,-82.776444
6539285,100935,20842,71.084006,1.628818,-69.455187
5206962,80422,19660,70.497461,1.380550,-69.116911
8008272,123746,1160,66.309922,1.696895,-64.613027
8251934,127577,5212,66.810306,2.260443,-64.549863
1704671,26489,19677,65.140046,1.882583,-63.257463
3616387,55989,48642,65.584609,3.118240,-62.466369
9094985,140753,28156,61.985765,0.700670,-61.285095
3850550,59742,7916,62.992056,1.971723,-61.020332
8659533,134057,45603,62.647444,2.226914,-60.420530


In [22]:
comparison[['user_id', 'product_id', 'recency_score', 'tfidf_score', 'diff', 'ratio']].equals("user_id == 1")

False

### Observations
- Products with **large positive diff** are highly distinctive for the user but not common overall.  
- Products with **large negative diff** are frequent, everyday items purchased by many users.  
- The TF–IDF model therefore emphasizes *personal relevance*, while Recency focuses on *frequency and recency of purchases*.  

The two methods **recommend different products**.  
- **Recency** favors frequent and recent purchases (everyday items).  
- **TF–IDF** favors rare or distinctive products that define the user’s unique taste.  
Combining them helps capture both **habitual** and **personal** preferences.

In [8]:
orders = pd.read_csv('../../data/orders.csv')
order_products = pd.read_csv('../../data/order_products__prior.csv')

orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [15]:
merged = order_products.merge(orders, on='order_id', how='inner')
print(merged.shape)
merged.head()

(32434489, 10)


,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2,33120,1,1,202279,prior,3,5,9,8.0
1,2,28985,2,1,202279,prior,3,5,9,8.0
2,2,9327,3,0,202279,prior,3,5,9,8.0
3,2,45918,4,1,202279,prior,3,5,9,8.0
4,2,30035,5,0,202279,prior,3,5,9,8.0


In [23]:
merged.query("user_id==10")

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
645758,68288,46979,1,1,10,prior,2,5,15,30.0
645759,68288,47380,2,0,10,prior,2,5,15,30.0
645760,68288,20995,3,0,10,prior,2,5,15,30.0
645761,68288,43014,4,0,10,prior,2,5,15,30.0
645762,68288,15011,5,0,10,prior,2,5,15,30.0
...,...,...,...,...,...,...,...,...,...,...
20057760,2115522,30489,29,1,10,prior,3,3,19,12.0
20057761,2115522,11068,30,0,10,prior,3,3,19,12.0
20057762,2115522,47526,31,1,10,prior,3,3,19,12.0
20057763,2115522,42736,32,0,10,prior,3,3,19,12.0
